# PortBench: learned ranker + Gemma 4 E4B re-ranking

Second stage of the two-stage localization in the PortBench paper (code and data: https://github.com/zak01E/portbench).

A logistic ranker, trained with project-level cross-validation, shortlists 30 files per instance (`data/rerank_inputs.json`, made by `learn_rank.py` and `export_rerank_inputs.py`). Gemma 4 E4B (4-bit NF4, greedy, thinking off, at most 350 new tokens) sees the target, the 30 paths in the ranker's order and a short excerpt of each file (#include lines and platform lines), and returns the files that need changes. Scoring is done by `gemma_rerank_analyze.py` in the repository.

Attach the model `google/gemma-4/transformers/gemma-4-e4b-it` as an input and use a T4 GPU.

In [ ]:
!pip install -q -U transformers accelerate kagglehub bitsandbytes
!wget -q https://raw.githubusercontent.com/zak01E/portbench/master/data/rerank_inputs.json
!wget -q https://raw.githubusercontent.com/zak01E/portbench/master/gemma_baseline.py
!wget -q https://raw.githubusercontent.com/zak01E/portbench/master/gemma_rerank.py
!grep -c 'chunked prefill' gemma_baseline.py
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import kagglehub
path = kagglehub.model_download('google/gemma-4/transformers/gemma-4-e4b-it')
print(path)

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True GEMMA_OUT=/kaggle/working/gemma_e4b_rerank.json python gemma_rerank.py rerank_inputs.json {path}

In [ ]:
import json
res = json.load(open('/kaggle/working/gemma_e4b_rerank.json'))
print(len(res), 'instances; empty answers:', sum(r['raw'] == '' for r in res))